# KSPP pseudopotentials — minimal SCF

Two short SCF workflows with automatic pseudopotential resolution from [KSPP](https://github.com/Quantum-MultiScale/KSPP):

1. **Driver** on FCC Al (`QEInput` + `ksppresolver=True`)
2. **QEpyCalculator** on rocksalt NaCl (two species, same API)

Requires a working QEpy + QE installation.

In [1]:
from pathlib import Path

from ase.build import bulk
from qepy.calculator import QEpyCalculator
from qepy.driver import Driver
from qepy.io import QEInput

NOTEBOOK_DIR = Path(".").resolve()

## SCF with `Driver` (Aluminum)

Pass a `QEInput` with `ksppresolver=True`; KSPP fills `pseudo_dir` and `atomic_species` before the run.

In [2]:
atoms_al = bulk("Al", "fcc", a=4.05, cubic=True)

qe_options_al = {
    "&control": {"calculation": "'scf'"},
    "&system": {
        "ibrav": 0,
        "degauss": 0.005,
        "ecutwfc": 30,
        "occupations": "'smearing'",
    },
    "&electrons": {"mixing_beta": 0.5},
    "k_points gamma": [],
}

pwin = QEInput(qe_options=qe_options_al, atoms=atoms_al, ksppresolver=True)
driver = Driver(pwin, atoms=atoms_al, logfile=NOTEBOOK_DIR / "al_scf.out")
driver.scf()

if driver.is_root:
    print("atomic_species:", pwin.qe_options["atomic_species"])
    print("converged:", driver.check_convergence())
    print("energy (Ry):", driver.get_energy())

driver.stop()

atomic_species: ['Al    26.981538 al_pbe_v1.uspp.F.UPF']
converged: True
energy (Ry): -26.33516633473628


## SCF with `QEpyCalculator` (NaCl)

Same `ksppresolver=True` flag on the ASE calculator interface.

In [3]:
atoms_nacl = bulk("NaCl", "rocksalt", a=5.64, cubic=True)

qe_options_nacl = {
    "&control": {"calculation": "'scf'"},
    "&system": {
        "ibrav": 0,
        "degauss": 0.02,
        "ecutwfc": 40,
        "occupations": "'smearing'",
    },
    "&electrons": {"mixing_beta": 0.5},
    "k_points gamma": [],
}

atoms_nacl.calc = QEpyCalculator(
    atoms=atoms_nacl,
    qe_options=qe_options_nacl,
    ksppresolver=True,
    logfile=NOTEBOOK_DIR / "nacl_scf.out",
)

energy = atoms_nacl.get_potential_energy()

if atoms_nacl.calc.is_root:
    print("atomic_species:", atoms_nacl.calc.qe_options["atomic_species"])
    print("energy (Ry):", energy)

atomic_species: ['Na    22.989769 na_pbe_v1.5.uspp.F.UPF', 'Cl    35.450000 cl_pbe_v1.4.uspp.F.UPF']
energy (Ry): -7013.1433609573705


## Further options

Configure the resolver before or after construction:

```python
pwin.ksppresolver(xc="LDA", offline=True, search_paths=[kspp_root])
calc.ksppresolver(table="norm-conserving/nc-sr-04", accuracy="stringent", update_ecuts=True)
```

KSPP reads suggested cutoffs from the UPF and warns when `ecutwfc` / `ecutrho` are too low; pass `update_ecuts=True` to raise them automatically.

For a fixed UPF on disk, set `pseudo_dir` and `atomic_species` explicitly in `qe_options` (see `examples/jupyter/DATA/`).